# AgentsVille AI Travel Planning Agent

This portfolio notebook demonstrates an agentic AI travel-planning workflow using structured output validation, weather-aware evaluation, tool descriptions, and a ReAct-style itinerary revision loop.

The production notebook used mocked course APIs for weather and activity data. This cleaned version keeps the core AI engineering artifacts visible and removes local secrets.

## 1. Environment-safe client configuration

The notebook reads provider settings from environment variables instead of hard-coding credentials.

In [ ]:
import os
from openai import OpenAI

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it to your local notebook environment before running LLM calls."
    )

client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)


## 2. Schema-first data modeling

Pydantic models make the LLM output inspectable, parseable, and testable.

In [ ]:
import datetime
from enum import Enum
from typing import List
from pydantic import BaseModel


class Interest(str, Enum):
    ART = "art"
    COOKING = "cooking"
    HIKING = "hiking"
    MUSIC = "music"
    PHOTOGRAPHY = "photography"
    TECHNOLOGY = "technology"
    TENNIS = "tennis"
    WRITING = "writing"


class Traveler(BaseModel):
    name: str
    age: int
    interests: List[Interest]


class VacationInfo(BaseModel):
    travelers: List[Traveler]
    destination: str
    date_of_arrival: datetime.date
    date_of_departure: datetime.date
    budget: int


class Weather(BaseModel):
    temperature: int
    unit: str
    condition: str


class Activity(BaseModel):
    activity_id: str
    name: str
    description: str
    start_time: str
    end_time: str
    price: int
    related_interests: List[Interest]


class ActivityRecommendation(BaseModel):
    activity: Activity
    reason: str


class ItineraryDay(BaseModel):
    date: datetime.date
    weather: Weather
    activity_recommendations: List[ActivityRecommendation]


class TravelPlan(BaseModel):
    city: str
    start_date: datetime.date
    end_date: datetime.date
    total_cost: int
    itinerary_days: List[ItineraryDay]


## 3. Initial itinerary agent prompt

The first itinerary-generation prompt keeps the stable trip details, weather catalog, activity catalog, and output schema together in the system context. The schema is injected as readable JSON rather than a Python dictionary representation.

In [ ]:
import json


def build_itinerary_agent_system_prompt(
    vacation_info: VacationInfo,
    weather_for_dates: list[dict],
    activities_for_dates: list[dict],
) -> str:
    """Build the system prompt for the first itinerary-generation pass."""

    return f"""
You are an expert travel agent specializing in itinerary planning and trip optimization.
Your job is to build realistic, enjoyable, and efficient itineraries by combining the travel calendar,
destination weather, activity availability, operating dates and hours, travel times, reservations,
and the user's preferences. You reason day by day, ensuring activities are scheduled only when they
are available, weather conditions are appropriate, and the overall pace of the trip remains comfortable.

## Task

* Create a day-by-day itinerary for the trip.
* Schedule outdoor activities only on days with suitable weather.
* Match activities to traveler interests.
* Stay within the trip budget.
* Include at least one activity recommendation per day.
* Calculate `total_cost` as the exact sum of all recommended activity prices.
* Use only activities from the provided activity catalog.

## Output format

ANALYSIS:
Briefly explain how you selected the city, dates, activities, weather-aware scheduling decisions,
traveler-interest matches, and budget tradeoffs.

FINAL OUTPUT:
Return a valid JSON object that conforms to this Pydantic schema:

{json.dumps(TravelPlan.model_json_schema(), indent=2)}

Do not include Markdown code fences.

## Context

VacationInfo trip details:

{vacation_info.model_dump_json(indent=2)}

Weather data:

{json.dumps(weather_for_dates, indent=2, default=str)}

Available activities data:

{json.dumps(activities_for_dates, indent=2, default=str)}
""".strip()


## 4. Weather compatibility evaluator prompt

This prompt uses constrained labels so evaluator output can be consumed by programmatic checks.

In [ ]:
ACTIVITY_AND_WEATHER_ARE_COMPATIBLE_SYSTEM_PROMPT = """
You are an expert travel agent and planner specializing in connecting travelers with activities that align with their interests.

## Task
Follow these rules:

* Default to compatible when evidence is limited: If the available activity or weather information is incomplete, unclear, or missing, classify the activity as IS_COMPATIBLE with the weather.
* Check for weather-safe backup options: Review the activity description for alternate plans, indoor options, rescheduling notes, or backup activities, and use those when deciding weather compatibility.

## Output format

    REASONING:
    Briefly explain the reasoning behind why the activity is or is not compatible with the weather.

    FINAL ANSWER:
    Return exactly one of:
    - IS_COMPATIBLE
    - IS_INCOMPATIBLE

    Do not include any other text after the final answer.

## Examples
* Serve & Savor: Tennis and Taste Luncheon IS_COMPATIBLE with (75, Fahrenheit, sunny)
* Trails & Tales: Lunchtime Hiking and Writing Retreat IS_INCOMPATIBLE with (-20, Celsius, hail)
""".strip()


## 5. Tool description for agent use

The docstring is written so it can be converted into an LLM-readable tool description.

In [ ]:
def get_activities_by_date_tool(date: str, city: str) -> List[dict]:
    """Fetch and validate available activities for a given city and date.

    Calls the activities API with the provided date and city, validates each
    returned activity against the `Activity` Pydantic model, and returns the
    validated activities as plain dictionaries for downstream AI/tool workflows.

    Args:
        date: Activity date in `YYYY-MM-DD` format.
        city: City where activities should be searched.

    Returns:
        A list of dictionaries, where each dictionary represents a validated
        activity returned by the activities API.

    Raises:
        pydantic.ValidationError: If any returned activity does not match the
            expected `Activity` schema.
    """
    raise NotImplementedError(
        "Connect this tool to the mocked project activity API in project_lib.py."
    )


## 6. ReAct revision agent system prompt

The revision agent cannot return a final answer until it has run validation. The action format is constrained to a single JSON object because the runner expects one tool call at a time.

In [ ]:
ITINERARY_REVISION_AGENT_SYSTEM_PROMPT = f"""
You are an expert travel agent specializing in itinerary review and revision.

## Task

Revise the provided TravelPlan so that it incorporates the traveler's feedback while still satisfying all itinerary constraints:

- Keep the destination, start date, and end date consistent with the original trip.
- Use only real activities returned by the available activity tools.
- Preserve weather-compatible scheduling.
- Keep the total cost accurate and within budget.
- Ensure the revised itinerary has at least two activities per day.
- Before returning the final itinerary, run `run_evals_tool`.
- Only call `final_answer_tool` after `run_evals_tool` reports success.

## ReAct Cycle

Use the following THINK-ACT-OBSERVE process:

1. THOUGHT:
   Reason about what needs to change in the itinerary.
   Decide whether you need to calculate a cost, fetch activities, run evaluations, or return the final answer.

2. ACTION:
   Output exactly one tool call as valid JSON using this format:
   {{"tool_name": "[tool_name]", "arguments": {{"arg1": "value1", "...": "..."}}}}

3. OBSERVATION:
   The system will execute your tool call and return an observation.
   Use that observation to decide the next THOUGHT and ACTION.

Repeat this cycle until the itinerary passes evaluation.

## Exit Instruction

You must call `run_evals_tool` on the revised TravelPlan before calling `final_answer_tool`.

If `run_evals_tool` returns failures, revise the itinerary and run `run_evals_tool` again.

Only when `run_evals_tool` returns success should you call:

{{"tool_name": "final_answer_tool", "arguments": {{"final_output": <valid TravelPlan JSON>}}}}

## Output Format

THOUGHT:
[brief reasoning about the next step]

ACTION:
{{"tool_name": "[tool_name]", "arguments": {{"arg1": "value1", "...": "..."}}}}

Rules:
- Output exactly one ACTION per response.
- The ACTION must be one JSON object, not a list or array.
- Start the ACTION line with `{{` and end it with `}}`.
- Do not wrap the ACTION in square brackets.
- The ACTION must be valid JSON.
- Do not include Markdown code fences.
- Do not include extra text after the ACTION.

## Available Tools

{get_tool_descriptions_string(ALL_TOOLS)}

## TravelPlan Schema

{json.dumps(TravelPlan.model_json_schema(), indent=2)}
""".strip()
